# Try IMSTE on a native single-cell dataset

The loader, embedding configuration, and plotting code live in `imste_single_cell.py`. This notebook keeps the experiment runnable in a few cells. Choose an original high-dimensional H5AD file from `data/` in the selector below.

Only original files are listed; PCA, scalp, and clasp variants are excluded. The loader uses the original `X` matrix and reads it in read-only mode. Set `MAX_CELLS` in the loading cell to choose a reproducible subset.


In [1]:
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys

import matplotlib.pyplot as plt

NOTEBOOK_DIR = next(
    (parent / "notebooks" for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "notebooks" / "imste_single_cell.py").is_file()),
    Path.cwd(),
)
if not (NOTEBOOK_DIR / "imste_single_cell.py").is_file():
    NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from imste_single_cell import (
    available_datasets,
    available_obs_columns,
    default_label_key,
    default_embedder_params,
    fit_embedding,
    load_dataset,
    plot_embedding,
    cross_validate_embedding_knn,
    plot_knn_confusion_matrix,
    plot_mst_edges,
    plot_class_neighborhood_hulls,
)
import pandas as pd


## Choose original dataset

Select one of the original high-dimensional H5AD files available in `data/`. PCA, scalp, and clasp variants are excluded. The default is the original human pancreas dataset.


In [ ]:
DEFAULT_DATASET_NAME = "human_pancreas_norm_complexBatch.h5ad"
AVAILABLE_DATASETS = available_datasets()
DATASET_OPTIONS = [(path.name, str(path)) for path in AVAILABLE_DATASETS]
DEFAULT_DATASET_PATH = next(
    value for name, value in DATASET_OPTIONS if name == DEFAULT_DATASET_NAME
)
try:
    import ipywidgets as widgets
    from IPython.display import display

    DATASET_SELECTOR = widgets.Dropdown(
        options=DATASET_OPTIONS,
        value=DEFAULT_DATASET_PATH,
        description="Dataset:",
        layout=widgets.Layout(width="80%"),
    )
    display(DATASET_SELECTOR)
except ImportError:
    DATASET_SELECTOR = None
    DATASET_NAME = DEFAULT_DATASET_NAME
    print("Available datasets:", ", ".join(path.name for path in AVAILABLE_DATASETS))
    print("Set DATASET_NAME to the file to load.")


In [ ]:
DATASET_PATH = (
    DATASET_SELECTOR.value
    if DATASET_SELECTOR is not None
    else next(path for path in AVAILABLE_DATASETS if path.name == DATASET_NAME)
)
MATRIX_KEY = "X"
MAX_CELLS = 3000
X, obs, row_indices = load_dataset(
    path=DATASET_PATH,
    matrix_key=MATRIX_KEY,
    obs_keys=tuple(available_obs_columns(DATASET_PATH)),
    max_cells=MAX_CELLS,
)
LABEL_KEY = default_label_key(obs)
print(f"Dataset: {Path(DATASET_PATH).name}")
print(f"Representation: {MATRIX_KEY}")
print(f"NumPy representation: {X.shape}, dtype={X.dtype}")
print(f"Rows loaded: {len(row_indices):,} / {X.shape[0]:,}")
print(f"Class label: {LABEL_KEY} ({obs[LABEL_KEY].nunique()} classes)")
print("Metadata columns:", list(obs.columns))
obs.head()


## Fit the embedder

The notebook calls the implementation in `imste_single_cell.py`. Edit `EMBEDDER_PARAMS` here to override the defaults defined in that file.


In [ ]:
EMBEDDER_PARAMS = default_embedder_params()
pd.Series(EMBEDDER_PARAMS, name="value").rename_axis("parameter").to_frame()

estimator, embedding = fit_embedding(X, EMBEDDER_PARAMS)


## Cross-validated kNN on the 2D embedding

Evaluate how well the 2D coordinates predict the selected `LABEL_KEY` using stratified cross-validation. The class-local hull plot uses only instances whose out-of-fold prediction matches their true label.


In [ ]:
KNN_NEIGHBORS = 5
CV_FOLDS = 5
knn_cv = cross_validate_embedding_knn(
    embedding, obs,
    label_key=LABEL_KEY,
    n_neighbors=KNN_NEIGHBORS,
    n_splits=CV_FOLDS,
)
display(pd.Series({
    "folds": knn_cv["n_splits"],
    "neighbors": knn_cv["n_neighbors"],
    "accuracy": knn_cv["accuracy"],
    "balanced accuracy": knn_cv["balanced_accuracy"],
}))
fig, ax = plot_knn_confusion_matrix(knn_cv)
plt.show()


In [ ]:
fig, axes = plot_embedding(embedding, obs, color_keys=(LABEL_KEY,))
plt.show()


## Try another stored representation or dataset

The same loader can read a dense or sparse expression matrix, or another AnnData representation. For example, set `matrix_key="X"` to load the stored expression matrix, or replace `DATA_FILE` with another `.h5ad` under `data/`. For large datasets, use `max_cells=3000` for a quick subset while keeping the original file untouched. IMSTE requires a dense NumPy array, so sparse inputs are densified only after selecting rows.

## Iterated MSTs on the 2D embedding

This cell builds fresh, exact, edge-disjoint MSTs directly from the fitted 2D coordinates. Change `N_TREES_TO_PLOT` to control the number of trees. Edges blend the endpoint colors for `LABEL_KEY` and use alpha `1 / rank`; node markers are omitted.


In [ ]:
N_TREES_TO_PLOT = 64
fig, ax = plot_mst_edges(
    embedding, obs,
    n_trees=N_TREES_TO_PLOT,
    color_key=LABEL_KEY,
)
plt.show()


## Class-local neighborhood hulls

For each correctly classified cell, build the convex hull of that cell and its `k` nearest correctly classified neighbors from the same class, then fill the union of those hulls using the class color. Set `K_NEIGHBORS` and `HULL_ALPHA` in the cell below.


In [ ]:
K_NEIGHBORS = 5
HULL_ALPHA = 0.5
fig, ax = plot_class_neighborhood_hulls(
    embedding, obs,
    k=K_NEIGHBORS,
    color_key=LABEL_KEY,
    alpha=HULL_ALPHA,
    include_mask=knn_cv["correct_mask"],
)
plt.show()
